# Monte Carlo generation of Manning roughness coefficient combinations

Case study: **Besaya River — Corrales de Buelna** (EPSG:25830)

Objective: build 1,000 stochastic combinations of Manning coefficients
for 9 land-use classes, based on bibliographic values (Chow 1959,
USGS, FHWA, Brater & King, Barnes).  
Each combination is used as input to an independent hydraulic simulation
(SFINCS or HEC-RAS).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

from pyhydra.modeling.hydraulic.sensitivity import generate_manning_combinations

## Paths — adjust for your environment

In [ ]:
DATA_DIR   = Path("/Volumes/My Passport 2/OneDrive/Scripts_Python/Paper_Rugosidades")
OUTPUT_DIR = Path("/Volumes/My Passport 2/COPIA_IH/E/Rugosidades_UCLM/Ejemplo_Besaya/nsim_rugos")

MANNING_DIST_CSV    = DATA_DIR / "manning_roughness_coefficients_dist.csv"
COMBINATIONS_CSV    = DATA_DIR / "combinaciones_rugosidad.csv"
MANNING_BASE_CSV    = Path("/Volumes/My Passport 2/COPIA_IH/E/Rugosidades_UCLM/Ejemplo_Besaya/manning.csv")

N_SAMPLES = 1000
SEED      = 42

_ext_ok = DATA_DIR.exists()

## 1 · Bibliographic Manning values by land-use class

In [ ]:
if _ext_ok:
    df_dist = pd.read_csv(MANNING_DIST_CSV, index_col=0)
    df_dist.head()

## 2 · Distribution fitting and Monte Carlo simulation

For each land-use class, the best-fit distribution among normal, lognormal,
and gamma is selected (criterion: highest KS p-value), 10,000 samples are
drawn, and 1,000 are selected without replacement.

In [ ]:
if _ext_ok:
    combinaciones_df = generate_manning_combinations(
        manning_dist_csv=str(MANNING_DIST_CSV),
        n_samples=N_SAMPLES,
        mc_size=10_000,
        seed=SEED,
    )
    print(f"Combinaciones generadas: {combinaciones_df.shape}")
    combinaciones_df.head()

## 3 · Fitted distribution visualisation

In [ ]:
if _ext_ok:
    from pyhydra.modeling.hydraulic.sensitivity import _best_distribution

    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    axes = axes.flatten()

    plot_idx = 0
    for _, row in df_dist.iterrows():
        if str(row["N"]) == "-999":
            continue
        values = np.array([float(v) for v in str(row["N"]).split(",")])
        land_use = row["Descripción"]

        best_dist, best_params = _best_distribution(values)
        dist = getattr(stats, best_dist)

        ax = axes[plot_idx]
        ax.hist(values, bins=5, density=True, alpha=0.5, color="steelblue", label="Bibliografía")
        ax.hist(combinaciones_df[land_use], bins=40, density=True, alpha=0.5,
                color="forestgreen", label="Monte Carlo")

        x = np.linspace(values.min() * 0.8, values.max() * 1.2, 200)
        pdf = dist.pdf(x, *best_params[:-2], loc=best_params[-2], scale=best_params[-1])
        ax.plot(x, pdf, "r-", lw=1.5, label=f"{best_dist}")

        ax.set_title(land_use, fontsize=9)
        ax.set_xlabel("n Manning", fontsize=8)
        ax.legend(fontsize=6)
        plot_idx += 1

    plt.suptitle("Fitted distributions and Monte Carlo samples by land use", y=1.01)
    plt.tight_layout()
    plt.show()

## 4 · Save combination matrix and per-simulation CSVs

In [ ]:
if _ext_ok:
    # Save master combinations table
    combinaciones_df.to_csv(COMBINATIONS_CSV, index=False)
    print(f"Guardado: {COMBINATIONS_CSV}")

In [ ]:
if _ext_ok:
    # Load Manning base table (landuse_code → description mapping)
    df_base = pd.read_csv(MANNING_BASE_CSV)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # The last column of combinaciones_df is NoData → N = -9999
    for i in range(N_SAMPLES):
        n_values = combinaciones_df.iloc[i].tolist() + [-9999]
        out = pd.DataFrame({
            "manning":     df_base["manning"].tolist(),
            "description": df_base["description"].tolist(),
            "landuse":     df_base["landuse"].tolist(),
            "N":           n_values,
        })
        out.to_csv(OUTPUT_DIR / f"combinacion_{i + 1}.csv", index=False)

    print(f"Generados {N_SAMPLES} CSVs en {OUTPUT_DIR}")

## 5 · Statistics of the generated combinations

In [ ]:
if _ext_ok:
    combinaciones_df.describe().round(4)

In [ ]:
if _ext_ok:
    fig, ax = plt.subplots(figsize=(10, 4))
    combinaciones_df.boxplot(ax=ax)
    ax.set_ylabel("n Manning")
    ax.set_title("Monte Carlo combination distribution by land use")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()